**02 Data Validation**

Source:
Harvard Growth Lab at Harvard University, The. 2025. “Bilateral Trade Data Aggregated by Year.” Harvard Dataverse. https://doi.org/10.7910/DVN/5NGVOB.
https://dataverse.harvard.edu/citation?persistentId=doi:10.7910/DVN/5NGVOB

**General Objective**
This code will explore and validate data from S2 1976-2023 Dataset
Global Trade Dataset Growth Lab

**1. Dataset general exploration**

In [ ]:
# --- 0) (Optional) install packages in THIS notebook kernel ---
# If you already installed them in your environment, this will just do nothing.
import sys
import subprocess

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure("duckdb")
ensure("pandas")

# --- 1) imports ---
import os
import duckdb
import pandas as pd
from IPython.display import display  # ensures display() exists in VS Code notebooks

# --- 2) paths / parameters ---
# IMPORTANT on Windows: use r"..." or double backslashes
BASE = r"C:\Python\trade\dataverse_files"

H0_BASELINE_YEAR = 1992  # pick an H0 year you know exists
S2_START_YEAR, S2_END_YEAR = 1976, 2023

EXPECTED_COLS = ["exporter", "importer", "commoditycode", "value_exporter", "value_importer"]

def describe_parquet(con, path):
    # schema introspection (lightweight)
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

def quick_sample(con, path, n=5):
    return con.execute(f"SELECT * FROM read_parquet('{path}') LIMIT {n}").df()

# --- 3) duckdb connection ---
con = duckdb.connect()
con.execute("PRAGMA threads=8")  # optional: use more CPU cores

# -------------------------
# 1) Get H0 baseline schema
# -------------------------
h0_path = os.path.join(BASE, f"H0_{H0_BASELINE_YEAR}.parquet")
if not os.path.exists(h0_path):
    con.close()
    raise FileNotFoundError(f"Baseline H0 file not found: {h0_path}")

h0_sch = describe_parquet(con, h0_path)
h0_types = h0_sch.set_index("column_name")["column_type"].to_dict()
h0_cols = h0_sch["column_name"].tolist()

print("H0 baseline file:", h0_path)
print("H0 baseline #cols:", len(h0_cols))
print("H0 baseline expected-cols present?:", all(c in set(h0_cols) for c in EXPECTED_COLS))

# -------------------------
# 2) Check S2 files vs H0
# -------------------------
rows = []
missing_files = []
missing_cols = []
type_mismatches = []
extra_cols = []
missing_relative_to_h0 = []

for y in range(S2_START_YEAR, S2_END_YEAR + 1):
    s2_path = os.path.join(BASE, f"S2_{y}.parquet")
    if not os.path.exists(s2_path):
        missing_files.append(y)
        rows.append({"year": y, "status": "missing_file"})
        continue

    try:
        s2_sch = describe_parquet(con, s2_path)
        s2_cols = s2_sch["column_name"].tolist()
        s2_types = s2_sch.set_index("column_name")["column_type"].to_dict()

        # required columns check
        miss_req = [c for c in EXPECTED_COLS if c not in set(s2_cols)]
        if miss_req:
            missing_cols.append((y, miss_req))

        # compare to H0 baseline: missing/extra columns (strict structure check)
        miss_vs_h0 = [c for c in h0_cols if c not in set(s2_cols)]
        extra_vs_h0 = [c for c in s2_cols if c not in set(h0_cols)]
        if miss_vs_h0:
            missing_relative_to_h0.append((y, miss_vs_h0))
        if extra_vs_h0:
            extra_cols.append((y, extra_vs_h0))

        # type mismatches for shared columns
        mism = []
        for c in set(h0_cols).intersection(set(s2_cols)):
            if h0_types.get(c) != s2_types.get(c):
                mism.append((c, h0_types.get(c), s2_types.get(c)))
        if mism:
            type_mismatches.append((y, mism))

        # Optional sanity sample (comment out to be schema-only)
        _ = quick_sample(con, s2_path, n=5)

        rows.append({
            "year": y,
            "status": "ok",
            "n_cols": len(s2_cols),
            "has_expected_cols": (len(miss_req) == 0),
            "missing_vs_h0": len(miss_vs_h0),
            "extra_vs_h0": len(extra_vs_h0),
            "type_mismatch_count": len(mism),
        })

    except Exception as e:
        rows.append({"year": y, "status": "error", "error": str(e)})

con.close()

report = pd.DataFrame(rows).sort_values("year")

print("\nMissing S2 files:", missing_files[:30], "..." if len(missing_files) > 30 else "")
print("\nYears missing REQUIRED cols (first 10):")
print(missing_cols[:10], "..." if len(missing_cols) > 10 else "")
print("\nYears with TYPE mismatches vs H0 (first 5):")
print(type_mismatches[:5], "..." if len(type_mismatches) > 5 else "")
print("\nYears with columns missing relative to H0 baseline (first 5):")
print(missing_relative_to_h0[:5], "..." if len(missing_relative_to_h0) > 5 else "")
print("\nYears with extra columns relative to H0 baseline (first 5):")
print(extra_cols[:5], "..." if len(extra_cols) > 5 else "")

display(report)


H0 baseline file: C:\Python\trade\dataverse_files\H0_1992.parquet
H0 baseline #cols: 7
H0 baseline expected-cols present?: True


**2. Data exploration**
First part of the code analyze content varaibles for all datasets
Second part of the code anallyze if all the datasets have the same variables

In [ ]:
# --- 0) Optional: install packages into THIS notebook kernel ---
import sys, subprocess

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure("duckdb")
ensure("pandas")

# --- 1) imports ---
import os
import duckdb
import pandas as pd
from IPython.display import display  # ensures display() works in VS Code notebooks

# --- 2) paths / params ---
BASE = r"C:\Python\trade\dataverse_files"   # IMPORTANT: raw string on Windows
S2_START_YEAR, S2_END_YEAR = 1976, 2023

# -------------------------
# Helper: map DuckDB types to broad categories
# -------------------------
def map_type(t: str) -> str:
    t = str(t).upper()
    if any(x in t for x in ["CHAR", "VARCHAR", "STRING", "TEXT"]):
        return "character"
    if any(x in t for x in ["TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT", "INT"]):
        return "integer"
    if any(x in t for x in ["DOUBLE", "FLOAT", "REAL", "DECIMAL", "NUMERIC"]):
        return "float"
    if any(x in t for x in ["DATE", "TIME", "TIMESTAMP"]):
        return "date/time"
    if "BOOL" in t:
        return "boolean"
    return "other"

def describe_parquet(con, path):
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

# -------------------------
# Collect schema info for all S2 files
# -------------------------
con = duckdb.connect()
con.execute("PRAGMA threads=8")  # optional speed-up

rows = []

for y in range(S2_START_YEAR, S2_END_YEAR + 1):
    path = os.path.join(BASE, f"S2_{y}.parquet")
    if not os.path.exists(path):
        continue

    sch = describe_parquet(con, path)

    for _, r in sch.iterrows():
        rows.append({
            "year": y,
            "variable": r["column_name"],
            "duckdb_type": r["column_type"],
            "broad_type": map_type(r["column_type"]),
        })

con.close()

schema_long = pd.DataFrame(rows).sort_values(["year", "variable"])

display(schema_long)


,year,variable,duckdb_type,broad_type
3,1976,commoditycode,VARCHAR,character
1,1976,exporter,VARCHAR,character
2,1976,importer,VARCHAR,character
5,1976,value_exporter,DOUBLE,float
4,1976,value_final,DOUBLE,float
...,...,...,...,...
331,2023,importer,VARCHAR,character
334,2023,value_exporter,DOUBLE,float
333,2023,value_final,DOUBLE,float
335,2023,value_importer,DOUBLE,float


In [ ]:
# --- 0) Optional: install packages into THIS notebook kernel ---
import sys, subprocess

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure("duckdb")
ensure("pandas")

# --- 1) imports ---
import os
import duckdb
import pandas as pd
from IPython.display import display  # ensures display() works in VS Code notebooks

# --- 2) paths / params ---
BASE = r"C:\Python\trade\dataverse_files"  # IMPORTANT: raw string on Windows
S2_START_YEAR, S2_END_YEAR = 1976, 2023

# -------------------------
# Helper: map DuckDB types to broad "nature"
# -------------------------
def map_nature(t: str) -> str:
    t = str(t).upper()

    if any(x in t for x in ["CHAR", "VARCHAR", "STRING", "TEXT"]):
        return "character"
    if any(x in t for x in ["TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT", "INT"]):
        return "integer"
    if any(x in t for x in ["DOUBLE", "FLOAT", "REAL", "DECIMAL", "NUMERIC"]):
        return "float"
    if any(x in t for x in ["DATE", "TIME", "TIMESTAMP"]):
        return "date/time"
    if "BOOL" in t:
        return "boolean"

    return "other"

def describe_parquet(con, path):
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

# -------------------------
# 1) Read schemas for all S2 files
# -------------------------
con = duckdb.connect()
con.execute("PRAGMA threads=8")  # optional speed-up

rows = []
missing_files = []

for y in range(S2_START_YEAR, S2_END_YEAR + 1):
    path = os.path.join(BASE, f"S2_{y}.parquet")
    if not os.path.exists(path):
        missing_files.append(y)
        continue

    sch = describe_parquet(con, path)
    for _, r in sch.iterrows():
        rows.append({
            "year": y,
            "variable": r["column_name"],
            "duckdb_type": r["column_type"],
            "nature": map_nature(r["column_type"]),
        })

con.close()

schema_long = pd.DataFrame(rows).sort_values(["year", "variable"])

if schema_long.empty:
    raise RuntimeError("No S2 parquet files were found/read. Check BASE path and filenames.")

# -------------------------
# 2) Define the canonical variable list (mode column set)
# -------------------------
cols_by_year = (
    schema_long.groupby("year")["variable"]
    .apply(lambda s: tuple(sorted(set(s))))
    .to_dict()
)

colset_counts = pd.Series(list(cols_by_year.values())).value_counts()
canonical_colset = set(colset_counts.index[0])

# If you prefer a specific baseline year instead, uncomment:
# BASELINE_YEAR = min(cols_by_year.keys())
# canonical_colset = set(cols_by_year[BASELINE_YEAR])

# -------------------------
# 3) Check: all years have the same variables
# -------------------------
missing_vars_by_year = {}
extra_vars_by_year = {}

for y, coltuple in cols_by_year.items():
    cset = set(coltuple)
    miss = sorted(canonical_colset - cset)
    extra = sorted(cset - canonical_colset)
    if miss:
        missing_vars_by_year[y] = miss
    if extra:
        extra_vars_by_year[y] = extra

# -------------------------
# 4) Check: variables have the same "nature" across years
# -------------------------
nature_by_var = (
    schema_long.groupby("variable")["nature"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="natures")
)

unstable_nature = nature_by_var[nature_by_var["natures"].apply(len) > 1].copy()

type_by_var = (
    schema_long.groupby("variable")["duckdb_type"]
    .apply(lambda x: sorted(set(x)))
    .reset_index(name="duckdb_types")
)

unstable_storage_type = type_by_var[type_by_var["duckdb_types"].apply(len) > 1].copy()

# -------------------------
# 5) Year-level summary table
# -------------------------
summary_rows = []
for y in sorted(cols_by_year.keys()):
    s = schema_long[schema_long["year"] == y]
    summary_rows.append({
        "year": y,
        "n_cols": s["variable"].nunique(),
        "missing_vs_canonical": len(missing_vars_by_year.get(y, [])),
        "extra_vs_canonical": len(extra_vars_by_year.get(y, [])),
    })

summary = pd.DataFrame(summary_rows)

print("Missing S2 files (first 30):", missing_files[:30], "..." if len(missing_files) > 30 else "")
print("\nCanonical column set size:", len(canonical_colset))
print("Years with missing vars vs canonical:", len(missing_vars_by_year))
print("Years with extra vars vs canonical:", len(extra_vars_by_year))
print("\nVariables with unstable NATURE across years:", len(unstable_nature))
print("Variables with unstable STORAGE TYPE across years:", len(unstable_storage_type))

display(summary)

if missing_vars_by_year:
    print("\nExample years missing variables (up to 5):")
    for y in list(missing_vars_by_year.keys())[:5]:
        print(y, "missing:", missing_vars_by_year[y][:20], "..." if len(missing_vars_by_year[y]) > 20 else "")

if extra_vars_by_year:
    print("\nExample years with extra variables (up to 5):")
    for y in list(extra_vars_by_year.keys())[:5]:
        print(y, "extra:", extra_vars_by_year[y][:20], "..." if len(extra_vars_by_year[y]) > 20 else "")

display(unstable_nature)
display(unstable_storage_type)

# -------------------------
# 6) Optional: enforce strictness (raise errors)
# -------------------------
STRICT_VARIABLES = False   # set True to fail if any year differs in variable set
STRICT_NATURE = True       # set True to fail if any variable changes nature across years

if STRICT_VARIABLES and (missing_vars_by_year or extra_vars_by_year):
    raise ValueError("Variable-set inconsistency detected across years (missing/extra columns). See diagnostics above.")

if STRICT_NATURE and not unstable_nature.empty:
    raise TypeError("Nature inconsistency detected (e.g., character vs numeric across years). See unstable_nature table above.")


Missing S2 files (first 30): [] 

Canonical column set size: 7
Years with missing vars vs canonical: 0
Years with extra vars vs canonical: 0

Variables with unstable NATURE across years: 0
Variables with unstable STORAGE TYPE across years: 0


,year,n_cols,missing_vs_canonical,extra_vs_canonical
0,1976,7,0,0
1,1977,7,0,0
2,1978,7,0,0
3,1979,7,0,0
4,1980,7,0,0
5,1981,7,0,0
6,1982,7,0,0
7,1983,7,0,0
8,1984,7,0,0
9,1985,7,0,0


,variable,natures


,variable,duckdb_types


**3. Dictionary**
This code builds a dictionary for codes, countries and products


In [ ]:
# --- 0) Optional: ensure duckdb is installed in THIS notebook kernel ---
import sys, subprocess

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure("duckdb")

# --- 1) imports ---
import os
import duckdb

# --- 2) paths ---
BASE = r"C:\Python\trade\dataverse_files"  # Windows-safe
OUT_DIR = os.path.join(BASE, "S02 dictionary")
os.makedirs(OUT_DIR, exist_ok=True)

S2_GLOB = os.path.join(BASE, "S2_*.parquet")

OUT_COUNTRIES = os.path.join(OUT_DIR, "country_codes_all_sorted.csv")
OUT_PRODUCTS  = os.path.join(OUT_DIR, "product_codes_all_sorted.csv")

# --- 3) build sorted dictionaries (overwrites existing files by default) ---
con = duckdb.connect()
con.execute("PRAGMA threads=8")

# A) All country codes (exporter ∪ importer), sorted
con.execute(f"""
COPY (
  WITH all_s2 AS (
    SELECT
      exporter::VARCHAR AS exporter_code,
      importer::VARCHAR AS importer_code
    FROM read_parquet('{S2_GLOB}')
  )
  SELECT DISTINCT code AS country_code
  FROM (
    SELECT exporter_code AS code FROM all_s2 WHERE exporter_code IS NOT NULL AND exporter_code <> ''
    UNION
    SELECT importer_code AS code FROM all_s2 WHERE importer_code IS NOT NULL AND importer_code <> ''
  )
  ORDER BY country_code
) TO '{OUT_COUNTRIES}' (HEADER, DELIMITER ',');
""")

# B) All product (commodity) codes, sorted
con.execute(f"""
COPY (
  SELECT DISTINCT commoditycode::VARCHAR AS product_code
  FROM read_parquet('{S2_GLOB}')
  WHERE commoditycode IS NOT NULL AND commoditycode <> ''
  ORDER BY product_code
) TO '{OUT_PRODUCTS}' (HEADER, DELIMITER ',');
""")

con.close()

print("Wrote (overwritten if existed) to:", OUT_DIR)
print(" -", os.path.basename(OUT_COUNTRIES))
print(" -", os.path.basename(OUT_PRODUCTS))


Wrote (overwritten if existed) to: C:\Python\trade\dataverse_files\S02 dictionary
 - country_codes_all_sorted.csv
 - product_codes_all_sorted.csv


**4. Cross Validation**
Export Import Cross Check Validation - Computational Intensive

In [ ]:
# =========================
# Symmetry audit for S2 trade panel (desktop / VS Code)
# =========================

import os
import sys
import subprocess
from datetime import datetime

def ensure(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

ensure("duckdb")
ensure("pandas")

import duckdb
import pandas as pd

# -------------------------
# User settings
# -------------------------
BASE = r"C:\Python\trade\dataverse_files"          # folder with S2_YYYY.parquet
OUT_DIR = os.path.join(BASE, "S02 dictionary")
os.makedirs(OUT_DIR, exist_ok=True)

S2_GLOB = os.path.join(BASE, "S2_*.parquet")
S2_START_YEAR, S2_END_YEAR = 1976, 2023

THREADS = 8

# Diagnostics thresholds
MIN_FLOW = 1e6     # only evaluate relative errors for flows with avg(X,M) >= MIN_FLOW
EPS = 1.0          # stabilizer in denominators, prevents division by zero

# “Top mismatches” output size
TOPK = 200

# Year used to infer symmetry mapping
SAMPLE_YEAR = 2019  # change if file missing

# Tolerance bands for “pass rates”
TOLS = [0.001, 0.01, 0.05]  # 0.1%, 1%, 5%

# -------------------------
# Helpers
# -------------------------
def rel_diff_sql(x, m):
    # symmetric relative difference: (x-m) / max(EPS, (x+m)/2)
    return f"(({x}) - ({m})) / GREATEST({EPS}, (({x}) + ({m})) / 2.0)"

def abs_rel_diff_sql(x, m):
    return f"ABS({rel_diff_sql(x,m)})"

def quantile_expr(col, q):
    # DuckDB quantile_cont
    return f"quantile_cont({col}, {q})"

def safe_year_exists():
    for y in range(S2_START_YEAR, S2_END_YEAR + 1):
        if os.path.exists(os.path.join(BASE, f"S2_{y}.parquet")):
            return y
    return None

# -------------------------
# Connect DuckDB
# -------------------------
con = duckdb.connect()
con.execute(f"PRAGMA threads={THREADS}")

# -------------------------
# 0) Determine symmetry mapping (same-row vs reversed-direction)
# -------------------------
# Same-row: compare sum(value_exporter) vs sum(value_importer) on same (a,b) or (a,b,c)
# Reversed: compare exports(a,b) vs imports(b,a) (requires join)
# We decide using bilateral (a,b) aggregated over products for a sample year.

if not os.path.exists(os.path.join(BASE, f"S2_{SAMPLE_YEAR}.parquet")):
    y0 = safe_year_exists()
    if y0 is None:
        con.close()
        raise RuntimeError("No S2_YYYY.parquet files found in BASE.")
    SAMPLE_YEAR = y0

sample_path = os.path.join(BASE, f"S2_{SAMPLE_YEAR}.parquet")
print("Using sample year for mapping:", SAMPLE_YEAR)

# Compute a robust “typical mismatch” for both interpretations on dyads with enough flow.
# Interpretation 1: same-row dyad compare X=sum(value_exporter) vs M=sum(value_importer) by (a,b)
same_row_stats = con.execute(f"""
WITH dyad AS (
  SELECT
    exporter,
    importer,
    SUM(value_exporter) AS X,
    SUM(value_importer) AS M
  FROM read_parquet('{sample_path}')
  WHERE value_exporter IS NOT NULL AND value_importer IS NOT NULL
  GROUP BY exporter, importer
),
filt AS (
  SELECT *,
    {abs_rel_diff_sql("X","M")} AS abs_rdiff,
    (X+M)/2.0 AS avg_flow
  FROM dyad
  WHERE (X+M)/2.0 >= {MIN_FLOW}
)
SELECT
  COUNT(*) AS n_dyads,
  {quantile_expr("abs_rdiff", 0.50)} AS med_abs_rdiff,
  {quantile_expr("abs_rdiff", 0.90)} AS p90_abs_rdiff,
  {quantile_expr("abs_rdiff", 0.99)} AS p99_abs_rdiff
FROM filt;
""").fetchdf()

# Interpretation 2: reversed-direction dyad compare X_ab vs M_ba (imports reported on reversed dyad)
rev_stats = con.execute(f"""
WITH dyad AS (
  SELECT
    exporter,
    importer,
    SUM(value_exporter) AS X
  FROM read_parquet('{sample_path}')
  WHERE value_exporter IS NOT NULL
  GROUP BY exporter, importer
),
dyad_m AS (
  SELECT
    exporter,
    importer,
    SUM(value_importer) AS M
  FROM read_parquet('{sample_path}')
  WHERE value_importer IS NOT NULL
  GROUP BY exporter, importer
),
joined AS (
  SELECT
    x.exporter AS exporter,
    x.importer AS importer,
    x.X AS X,
    m.M AS M_mirror
  FROM dyad x
  LEFT JOIN dyad_m m
    ON m.exporter = x.importer AND m.importer = x.exporter
),
filt AS (
  SELECT *,
    {abs_rel_diff_sql("X","M_mirror")} AS abs_rdiff,
    (X+COALESCE(M_mirror,0))/2.0 AS avg_flow
  FROM joined
  WHERE (X+COALESCE(M_mirror,0))/2.0 >= {MIN_FLOW}
)
SELECT
  COUNT(*) AS n_dyads,
  {quantile_expr("abs_rdiff", 0.50)} AS med_abs_rdiff,
  {quantile_expr("abs_rdiff", 0.90)} AS p90_abs_rdiff,
  {quantile_expr("abs_rdiff", 0.99)} AS p99_abs_rdiff
FROM filt;
""").fetchdf()

print("\nMapping diagnostic (dyad-level, sample year):")
display(pd.concat([
    same_row_stats.assign(mapping="same_row"),
    rev_stats.assign(mapping="reversed_direction")
], ignore_index=True))

# Choose mapping by median absolute relative diff (lower is “more symmetric”)
same_med = float(same_row_stats["med_abs_rdiff"].iloc[0]) if not same_row_stats.empty else float("inf")
rev_med  = float(rev_stats["med_abs_rdiff"].iloc[0]) if not rev_stats.empty else float("inf")
MAPPING = "same_row" if same_med <= rev_med else "reversed_direction"

print("\nChosen mapping:", MAPPING)

# -------------------------
# 1) Country-level test per year
# -------------------------
# For each country a:
#   X_a = sum exports attributed to a
#   M_a = sum mirror imports attributed to a (depends on mapping)
#
# same_row mapping:
#   exports by a: sum(value_exporter where exporter=a)
#   mirror imports of a: sum(value_importer where importer=a)  (same-row)
#
# reversed_direction mapping:
#   exports by a: sum(value_exporter where exporter=a)
#   mirror imports of a: sum(value_importer where exporter=a) ???  (careful)
#   Under reversed-direction, value_importer on (b,a) is the mirror of exports(a,b),
#   so imports attributed to a as "mirror of its exports" is sum over partners b of value_importer on (b,a),
#   which is sum(value_importer where importer=a). (Same as above for totals.)
#
# Net: for country totals, both mappings reduce to comparing:
#   sum(value_exporter by exporter=a) vs sum(value_importer by importer=a)
#
country_year = con.execute(f"""
WITH all_rows AS (
  SELECT
    regexp_extract(filename, 'S2_([0-9]{{4}})\\.parquet', 1)::INTEGER AS year,
    exporter,
    importer,
    value_exporter,
    value_importer
  FROM read_parquet('{S2_GLOB}', filename=true)
),
exports AS (
  SELECT year, exporter AS country, SUM(value_exporter) AS X
  FROM all_rows
  WHERE value_exporter IS NOT NULL
  GROUP BY year, exporter
),
imports AS (
  SELECT year, importer AS country, SUM(value_importer) AS M
  FROM all_rows
  WHERE value_importer IS NOT NULL
  GROUP BY year, importer
),
joined AS (
  SELECT
    COALESCE(e.year, i.year) AS year,
    COALESCE(e.country, i.country) AS country,
    COALESCE(e.X, 0) AS X,
    COALESCE(i.M, 0) AS M
  FROM exports e
  FULL OUTER JOIN imports i
    ON e.year=i.year AND e.country=i.country
),
scored AS (
  SELECT *,
    (X+M)/2.0 AS avg_flow,
    {rel_diff_sql("X","M")} AS rdiff,
    {abs_rel_diff_sql("X","M")} AS abs_rdiff
  FROM joined
)
SELECT * FROM scored;
""").fetchdf()

# Year summary for country test
def summarize(df, level_name):
    out = []
    for y, g in df.groupby("year"):
        g2 = g[g["avg_flow"] >= MIN_FLOW].copy()
        n = len(g2)
        if n == 0:
            out.append({"year": y, "level": level_name, "n": 0})
            continue
        row = {
            "year": y,
            "level": level_name,
            "n": n,
            "median_abs_rdiff": g2["abs_rdiff"].median(),
            "p90_abs_rdiff": g2["abs_rdiff"].quantile(0.90),
            "p99_abs_rdiff": g2["abs_rdiff"].quantile(0.99),
        }
        for tol in TOLS:
            row[f"share_within_{tol}"] = (g2["abs_rdiff"] <= tol).mean()
        out.append(row)
    return pd.DataFrame(out).sort_values("year")

country_year_summary = summarize(country_year, "country_totals")

# -------------------------
# 2) Bilateral dyad test per year (aggregated over products)
# -------------------------
if MAPPING == "same_row":
    dyad_year = con.execute(f"""
    WITH all_rows AS (
      SELECT
        regexp_extract(filename, 'S2_([0-9]{{4}})\\.parquet', 1)::INTEGER AS year,
        exporter,
        importer,
        value_exporter,
        value_importer
      FROM read_parquet('{S2_GLOB}', filename=true)
    ),
    dyad AS (
      SELECT
        year, exporter, importer,
        SUM(value_exporter) AS X,
        SUM(value_importer) AS M
      FROM all_rows
      GROUP BY year, exporter, importer
    )
    SELECT
      year, exporter, importer, X, M,
      (X+M)/2.0 AS avg_flow,
      {rel_diff_sql("X","M")} AS rdiff,
      {abs_rel_diff_sql("X","M")} AS abs_rdiff
    FROM dyad;
    """).fetchdf()
else:
    # reversed-direction mapping: compare exports(a,b) to importer-values on (b,a)
    dyad_year = con.execute(f"""
    WITH all_rows AS (
      SELECT
        regexp_extract(filename, 'S2_([0-9]{{4}})\\.parquet', 1)::INTEGER AS year,
        exporter,
        importer,
        value_exporter,
        value_importer
      FROM read_parquet('{S2_GLOB}', filename=true)
    ),
    x AS (
      SELECT year, exporter, importer, SUM(value_exporter) AS X
      FROM all_rows
      GROUP BY year, exporter, importer
    ),
    m AS (
      SELECT year, exporter, importer, SUM(value_importer) AS M
      FROM all_rows
      GROUP BY year, exporter, importer
    ),
    j AS (
      SELECT
        x.year,
        x.exporter,
        x.importer,
        x.X,
        COALESCE(m.M, 0) AS M_mirror
      FROM x
      LEFT JOIN m
        ON m.year = x.year AND m.exporter = x.importer AND m.importer = x.exporter
    )
    SELECT
      year, exporter, importer,
      X,
      M_mirror AS M,
      (X+M_mirror)/2.0 AS avg_flow,
      {rel_diff_sql("X","M_mirror")} AS rdiff,
      {abs_rel_diff_sql("X","M_mirror")} AS abs_rdiff
    FROM j;
    """).fetchdf()

dyad_year_summary = summarize(dyad_year, "bilateral_totals")

# -------------------------
# 3) Bilateral-by-product test per year
# -------------------------
# Warning: can be very large. We compute summary stats and only export TOPK mismatches per year (or overall).
if MAPPING == "same_row":
    dyad_prod = con.execute(f"""
    WITH all_rows AS (
      SELECT
        regexp_extract(filename, 'S2_([0-9]{{4}})\\.parquet', 1)::INTEGER AS year,
        exporter,
        importer,
        commoditycode,
        value_exporter,
        value_importer
      FROM read_parquet('{S2_GLOB}', filename=true)
    ),
    g AS (
      SELECT
        year, exporter, importer, commoditycode,
        SUM(value_exporter) AS X,
        SUM(value_importer) AS M
      FROM all_rows
      GROUP BY year, exporter, importer, commoditycode
    )
    SELECT
      year, exporter, importer, commoditycode, X, M,
      (X+M)/2.0 AS avg_flow,
      {rel_diff_sql("X","M")} AS rdiff,
      {abs_rel_diff_sql("X","M")} AS abs_rdiff
    FROM g;
    """).fetchdf()
else:
    dyad_prod = con.execute(f"""
    WITH all_rows AS (
      SELECT
        regexp_extract(filename, 'S2_([0-9]{{4}})\\.parquet', 1)::INTEGER AS year,
        exporter,
        importer,
        commoditycode,
        value_exporter,
        value_importer
      FROM read_parquet('{S2_GLOB}', filename=true)
    ),
    x AS (
      SELECT year, exporter, importer, commoditycode, SUM(value_exporter) AS X
      FROM all_rows
      GROUP BY year, exporter, importer, commoditycode
    ),
    m AS (
      SELECT year, exporter, importer, commoditycode, SUM(value_importer) AS M
      FROM all_rows
      GROUP BY year, exporter, importer, commoditycode
    ),
    j AS (
      SELECT
        x.year, x.exporter, x.importer, x.commoditycode,
        x.X,
        COALESCE(m.M, 0) AS M_mirror
      FROM x
      LEFT JOIN m
        ON m.year=x.year
       AND m.exporter=x.importer
       AND m.importer=x.exporter
       AND m.commoditycode=x.commoditycode
    )
    SELECT
      year, exporter, importer, commoditycode,
      X,
      M_mirror AS M,
      (X+M_mirror)/2.0 AS avg_flow,
      {rel_diff_sql("X","M_mirror")} AS rdiff,
      {abs_rel_diff_sql("X","M_mirror")} AS abs_rdiff
    FROM j;
    """).fetchdf()

dyad_prod_summary = summarize(dyad_prod, "bilateral_by_product")

# -------------------------
# 4) Export outputs
# -------------------------
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

country_year.to_csv(os.path.join(OUT_DIR, f"country_totals_scored_{stamp}.csv"), index=False)
dyad_year.to_csv(os.path.join(OUT_DIR, f"bilateral_totals_scored_{stamp}.csv"), index=False)

# Do NOT dump full dyad_prod by default (can be enormous). Export summary + top mismatches.
country_year_summary.to_csv(os.path.join(OUT_DIR, f"summary_country_{stamp}.csv"), index=False)
dyad_year_summary.to_csv(os.path.join(OUT_DIR, f"summary_bilateral_{stamp}.csv"), index=False)
dyad_prod_summary.to_csv(os.path.join(OUT_DIR, f"summary_bilateral_product_{stamp}.csv"), index=False)

# Top mismatches (overall) above MIN_FLOW
top_country = (
    country_year[country_year["avg_flow"] >= MIN_FLOW]
    .sort_values("abs_rdiff", ascending=False)
    .head(TOPK)
)
top_dyad = (
    dyad_year[dyad_year["avg_flow"] >= MIN_FLOW]
    .sort_values("abs_rdiff", ascending=False)
    .head(TOPK)
)
top_dyad_prod = (
    dyad_prod[dyad_prod["avg_flow"] >= MIN_FLOW]
    .sort_values("abs_rdiff", ascending=False)
    .head(TOPK)
)

top_country.to_csv(os.path.join(OUT_DIR, f"top_mismatch_country_{stamp}.csv"), index=False)
top_dyad.to_csv(os.path.join(OUT_DIR, f"top_mismatch_bilateral_{stamp}.csv"), index=False)
top_dyad_prod.to_csv(os.path.join(OUT_DIR, f"top_mismatch_bilateral_product_{stamp}.csv"), index=False)

# -------------------------
# 5) Generate a Markdown report
# -------------------------
def fmt_pct(x):
    if pd.isna(x): return ""
    return f"{100*x:.1f}%"

def summarize_block(df_summary, title):
    # take latest year row if exists
    if df_summary.empty:
        return f"{title}\n\nNo observations above MIN_FLOW.\n"
    latest = df_summary.sort_values("year").iloc[-1]
    lines = []
    lines.append(f"{title}\n")
    lines.append(f"- Coverage (year {int(latest['year'])}): n={int(latest.get('n',0))} entities above MIN_FLOW={MIN_FLOW:,.0f}")
    if "median_abs_rdiff" in latest:
        lines.append(f"- Median |rel diff|: {latest['median_abs_rdiff']:.4f}")
        lines.append(f"- p90 |rel diff|: {latest['p90_abs_rdiff']:.4f}")
        lines.append(f"- p99 |rel diff|: {latest['p99_abs_rdiff']:.4f}")
    for tol in TOLS:
        col = f"share_within_{tol}"
        if col in latest and pd.notna(latest[col]):
            lines.append(f"- Share within ±{tol*100:.1f}%: {fmt_pct(latest[col])}")
    return "\n".join(lines) + "\n"

report_path = os.path.join(OUT_DIR, f"Symmetry_Audit_Report_{stamp}.md")

with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"# Symmetry Audit Report (S2 trade data)\n\n")
    f.write(f"- Base folder: `{BASE}`\n")
    f.write(f"- Years: {S2_START_YEAR}–{S2_END_YEAR}\n")
    f.write(f"- MIN_FLOW used for relative-error diagnostics: {MIN_FLOW:,.0f}\n")
    f.write(f"- Mapping chosen: **{MAPPING}** (based on sample year {SAMPLE_YEAR})\n\n")

    f.write("## What was tested\n\n")
    f.write("1) Country totals: compare total exports of each country to mirror imports attributed to that country.\n")
    f.write("2) Bilateral totals: compare exports a→b to mirror imports b←a (aggregated over products).\n")
    f.write("3) Bilateral-by-product: compare exports a→b for product c to mirror imports b←a for product c.\n\n")

    f.write("## Results summary\n\n")
    f.write(summarize_block(country_year_summary, "### Country totals"))
    f.write("\n")
    f.write(summarize_block(dyad_year_summary, "### Bilateral totals (aggregated over products)"))
    f.write("\n")
    f.write(summarize_block(dyad_prod_summary, "### Bilateral-by-product"))
    f.write("\n")

    f.write("## Where mismatches concentrate\n\n")
    f.write(f"- Top {TOPK} mismatches were exported to CSV:\n")
    f.write(f"  - `top_mismatch_country_{stamp}.csv`\n")
    f.write(f"  - `top_mismatch_bilateral_{stamp}.csv`\n")
    f.write(f"  - `top_mismatch_bilateral_product_{stamp}.csv`\n\n")

    f.write("## Caveats and interpretation\n\n")
    f.write("- Relative differences become unstable for small flows; this is why MIN_FLOW is applied.\n")
    f.write("- If exporter vs importer values correspond to different valuation concepts (FOB vs CIF) or different reconciliation steps, exact equality is not expected.\n")
    f.write("- Missingness or suppression can create asymmetry mechanically (mirror cell absent/zero).\n")
    f.write("- Country code transitions (splits/mergers) can create concentrated mismatches in specific years.\n")
    f.write("- If you need a strict ‘symmetric matrix’ for modeling, you can construct a reconciled value (e.g., average of mirrors) after identifying problematic dyads.\n")

print("Done.")
print("Outputs written to:", OUT_DIR)
print("Markdown report:", report_path)
print("Top mismatch CSVs and summary CSVs written with timestamp:", stamp)

# Close DuckDB
con.close()

# Show quick views in notebook
display(country_year_summary.tail(10))
display(dyad_year_summary.tail(10))
display(dyad_prod_summary.tail(10))


Using sample year for mapping: 2019

Mapping diagnostic (dyad-level, sample year):


,n_dyads,med_abs_rdiff,p90_abs_rdiff,p99_abs_rdiff,mapping
0,16261,0.508495,2.000000,2.0,same_row
1,16527,1.331858,1.982334,2.0,reversed_direction



Chosen mapping: same_row
Done.
Outputs written to: C:\Python\trade\dataverse_files\S02 dictionary
Markdown report: C:\Python\trade\dataverse_files\S02 dictionary\Symmetry_Audit_Report_20260128_122806.md
Top mismatch CSVs and summary CSVs written with timestamp: 20260128_122806


,year,level,n,median_abs_rdiff,p90_abs_rdiff,p99_abs_rdiff,share_within_0.001,share_within_0.01,share_within_0.05
38,2014,country_totals,171,0.372848,1.522711,1.824018,0.005848,0.035088,0.076023
39,2015,country_totals,174,0.374991,1.578769,1.969325,0.005747,0.022989,0.091954
40,2016,country_totals,175,0.382407,1.507598,1.962265,0.000000,0.005714,0.062857
41,2017,country_totals,177,0.425903,1.574216,1.888972,0.000000,0.011299,0.062147
42,2018,country_totals,175,0.433606,1.579602,1.897649,0.000000,0.028571,0.057143
43,2019,country_totals,170,0.391145,1.443393,1.916680,0.005882,0.005882,0.029412
44,2020,country_totals,166,0.371051,1.582400,1.871833,0.006024,0.018072,0.048193
45,2021,country_totals,167,0.355298,1.567024,1.901743,0.000000,0.005988,0.071856
46,2022,country_totals,159,0.391048,1.560050,1.891023,0.000000,0.000000,0.037736
47,2023,country_totals,152,0.360499,1.590203,1.947700,0.000000,0.006579,0.085526


,year,level,n,median_abs_rdiff,p90_abs_rdiff,p99_abs_rdiff,share_within_0.001,share_within_0.01,share_within_0.05
38,2014,bilateral_totals,15952,0.567453,2.0,2.0,0.001818,0.014042,0.071715
39,2015,bilateral_totals,15880,0.530057,2.0,2.0,0.001196,0.013224,0.069899
40,2016,bilateral_totals,15844,0.490255,2.0,2.0,0.001704,0.014895,0.074918
41,2017,bilateral_totals,16170,0.501213,2.0,2.0,0.001855,0.014719,0.071367
42,2018,bilateral_totals,16221,0.510198,2.0,2.0,0.001233,0.012885,0.071204
43,2019,bilateral_totals,16261,0.508495,2.0,2.0,0.002029,0.015743,0.075272
44,2020,bilateral_totals,15849,0.516639,2.0,2.0,0.001010,0.013566,0.068774
45,2021,bilateral_totals,16376,0.546666,2.0,2.0,0.001588,0.016121,0.070286
46,2022,bilateral_totals,16521,0.646346,2.0,2.0,0.001332,0.012408,0.061497
47,2023,bilateral_totals,16300,0.688424,2.0,2.0,0.001104,0.011718,0.063988


,year,level,n,median_abs_rdiff,p90_abs_rdiff,p99_abs_rdiff,share_within_0.001,share_within_0.01,share_within_0.05
38,2014,bilateral_by_product,558762,0.443512,1.951527,2.0,0.001528,0.014851,0.074672
39,2015,bilateral_by_product,536733,0.430161,1.914332,2.0,0.001548,0.014868,0.074885
40,2016,bilateral_by_product,532361,0.419780,1.905532,2.0,0.001360,0.014721,0.073801
41,2017,bilateral_by_product,555565,0.416969,1.902904,2.0,0.001487,0.014722,0.075196
42,2018,bilateral_by_product,574142,0.417054,1.930638,2.0,0.001543,0.014892,0.074898
43,2019,bilateral_by_product,571238,0.412113,1.917063,2.0,0.001502,0.014866,0.074885
44,2020,bilateral_by_product,550117,0.412115,1.928413,2.0,0.001427,0.014873,0.073955
45,2021,bilateral_by_product,596236,0.420183,1.936253,2.0,0.001464,0.014568,0.072938
46,2022,bilateral_by_product,607662,0.453246,1.998423,2.0,0.001410,0.014541,0.071954
47,2023,bilateral_by_product,596812,0.474007,2.000000,2.0,0.001287,0.013671,0.069694


**5. Estimation of Import Export Data with value final**
Estimate total import and exports with adjunted value from Growth Lab

In [1]:
import duckdb
import pandas as pd
import os
from datetime import datetime

BASE = r"C:\Python\trade\dataverse_files"
S2_START_YEAR, S2_END_YEAR = 1976, 2023

# ---- quick sanity: list a few matching files
all_files = [f for f in os.listdir(BASE) if f.startswith("S2_") and f.endswith(".parquet")]
print("Found S2 parquet files:", len(all_files))
print("First 5:", all_files[:5])

if len(all_files) == 0:
    raise RuntimeError(f"No S2_*.parquet files found in {BASE}. Check the folder path.")

con = duckdb.connect()
con.execute("PRAGMA threads=8")

rows = []
missing_years = []
years_processed = 0
t0 = datetime.now()

for y in range(S2_START_YEAR, S2_END_YEAR + 1):
    path = os.path.join(BASE, f"S2_{y}.parquet")
    if not os.path.exists(path):
        missing_years.append(y)
        continue

    years_processed += 1
    if years_processed % 5 == 0:
        elapsed = (datetime.now() - t0).total_seconds()
        print(f"Processed {years_processed} files so far... (latest year {y}, elapsed {elapsed:.1f}s)")

    # Check that value_final exists in this file (fast schema check)
    sch = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()
    cols = set(sch["column_name"].tolist())
    if "value_final" not in cols:
        print(f"WARNING: {os.path.basename(path)} has no 'value_final'. Columns include: {sorted(list(cols))[:10]} ...")
        continue

    try:
        tmp = con.execute(f"""
            WITH exports AS (
                SELECT exporter AS country, SUM(value_final) AS exports
                FROM read_parquet('{path}')
                GROUP BY exporter
            ),
            imports AS (
                SELECT importer AS country, SUM(value_final) AS imports
                FROM read_parquet('{path}')
                GROUP BY importer
            )
            SELECT
                {y} AS year,
                COALESCE(e.country, i.country) AS country,
                COALESCE(e.exports, 0) AS exports,
                COALESCE(i.imports, 0) AS imports,
                COALESCE(e.exports, 0) - COALESCE(i.imports, 0) AS trade_balance
            FROM exports e
            FULL JOIN imports i
              ON e.country = i.country
        """).df()

        rows.append(tmp)

    except Exception as e:
        print(f"ERROR in year {y}: {e}")
        # keep going so you get a partial output instead of nothing
        continue

con.close()

print("Missing years (count):", len(missing_years))
print("Years processed (files found):", years_processed)
print("Rows tables collected:", len(rows))

if len(rows) == 0:
    raise RuntimeError("No yearly tables were produced. Most likely: no files found, or 'value_final' missing in all files.")

panel = pd.concat(rows, ignore_index=True)

print("Panel shape:", panel.shape)
print(panel.head(10))

OUT_FILE = r"C:\Python\trade\country_trade_totals_value_final.csv"
panel.to_csv(OUT_FILE, index=False)
print("Wrote:", OUT_FILE)
print("File exists now?:", os.path.exists(OUT_FILE))


Found S2 parquet files: 48
First 5: ['S2_1976.parquet', 'S2_1977.parquet', 'S2_1978.parquet', 'S2_1979.parquet', 'S2_1980.parquet']
Processed 5 files so far... (latest year 1980, elapsed 0.2s)
Processed 10 files so far... (latest year 1985, elapsed 0.5s)
Processed 15 files so far... (latest year 1990, elapsed 0.8s)
Processed 20 files so far... (latest year 1995, elapsed 1.0s)
Processed 25 files so far... (latest year 2000, elapsed 1.4s)
Processed 30 files so far... (latest year 2005, elapsed 1.8s)
Processed 35 files so far... (latest year 2010, elapsed 2.6s)
Processed 40 files so far... (latest year 2015, elapsed 3.4s)
Processed 45 files so far... (latest year 2020, elapsed 4.1s)
Missing years (count): 0
Years processed (files found): 48
Rows tables collected: 48
Panel shape: (10551, 5)
   year country       exports       imports  trade_balance
0  1976     ZMB  1.003951e+09  7.087314e+08   2.952191e+08
1  1976     BRA  9.879747e+09  1.209039e+10  -2.210644e+09
2  1976     CIV  1.719314